In [75]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split

In [76]:
#%cd ../..
# !ls

In [113]:
tmp_data = pd.read_csv('EDA/data/findata.csv', index_col=0)

In [114]:
tmp_data.info(show_counts=True, verbose=True)

<class 'pandas.core.frame.DataFrame'>
Index: 1001 entries, 0 to 1000
Data columns (total 79 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   IC50, mM             1001 non-null   float64
 1   CC50, mM             1001 non-null   float64
 2   SI                   1001 non-null   float64
 3   MaxAbsEStateIndex    1001 non-null   float64
 4   MaxEStateIndex       1001 non-null   float64
 5   MinAbsEStateIndex    1001 non-null   float64
 6   MinEStateIndex       1001 non-null   float64
 7   qed                  1001 non-null   float64
 8   SPS                  1001 non-null   float64
 9   MolWt                1001 non-null   float64
 10  HeavyAtomMolWt       1001 non-null   float64
 11  ExactMolWt           1001 non-null   float64
 12  NumValenceElectrons  1001 non-null   float64
 13  MaxPartialCharge     1001 non-null   float64
 14  MinPartialCharge     1001 non-null   float64
 15  MaxAbsPartialCharge  1001 non-null   float6

In [115]:
go_data = tmp_data.copy()

In [116]:
# Удалим не нужные тут таргеты 
go_data = go_data.drop(columns=['CC50, mM', 'SI'])

In [169]:
# Выбираем самые полезные параметры 

# Рассчитываем корреляцию всех признаков 
go_correlations = go_data.corr()['IC50, mM'].abs().sort_values()

# Отбираем признаки с корреляцией больше 0.1 
gl_high_info_features = go_correlations[go_correlations > 0.1]

print("Информативные признаки (есть связь):")
gl_high_info_features = gl_high_info_features.drop(['IC50, mM'], errors='ignore')


#Для финальной модели оставляем только информативные параметры  
gl_final_param = gl_high_info_features.index.unique().tolist() 

print(len(gl_final_param))
display(go_correlations.sort_values(ascending=False).head(50))

display(gl_final_param)

Информативные признаки (есть связь):
47


IC50, mM               1.000000
PEOE_VSA7              0.266011
VSA_EState4            0.262503
Chi2n                  0.257058
Chi2v                  0.249164
Chi4v                  0.243600
Chi4n                  0.243497
Chi3n                  0.239741
Chi3v                  0.237759
Chi1n                  0.229828
Chi1v                  0.219810
MolLogP                0.218387
MolMR                  0.215792
Chi0n                  0.215641
Chi0v                  0.212552
BCUT2D_LOGPHI          0.209914
FpDensityMorgan1       0.208710
SlogP_VSA5             0.205577
BCUT2D_CHGLO           0.203161
BalabanJ               0.196842
SMR_VSA5               0.196466
MinEStateIndex         0.188243
FpDensityMorgan2       0.187991
LabuteASA              0.186695
PEOE_VSA6              0.182623
Kappa1                 0.180523
MaxPartialCharge       0.179227
NumValenceElectrons    0.178152
VSA_EState7            0.170529
HeavyAtomCount         0.167852
Kappa2                 0.167696
Chi0    

['BCUT2D_LOGPLOW',
 'MaxEStateIndex',
 'MaxAbsEStateIndex',
 'EState_VSA4',
 'HeavyAtomMolWt',
 'Kappa3',
 'MinAbsPartialCharge',
 'RingCount',
 'VSA_EState8',
 'NumRotatableBonds',
 'FpDensityMorgan3',
 'BCUT2D_MWLOW',
 'ExactMolWt',
 'MolWt',
 'EState_VSA8',
 'Chi1',
 'Chi0',
 'Kappa2',
 'HeavyAtomCount',
 'VSA_EState7',
 'NumValenceElectrons',
 'MaxPartialCharge',
 'Kappa1',
 'PEOE_VSA6',
 'LabuteASA',
 'FpDensityMorgan2',
 'MinEStateIndex',
 'SMR_VSA5',
 'BalabanJ',
 'BCUT2D_CHGLO',
 'SlogP_VSA5',
 'FpDensityMorgan1',
 'BCUT2D_LOGPHI',
 'Chi0v',
 'Chi0n',
 'MolMR',
 'MolLogP',
 'Chi1v',
 'Chi1n',
 'Chi3v',
 'Chi3n',
 'Chi4n',
 'Chi4v',
 'Chi2v',
 'Chi2n',
 'VSA_EState4',
 'PEOE_VSA7']

In [170]:
#перебором выявим параметры котрые коррелируют между собой > 60% и оставим только второй 

for col_1 in gl_final_param:
    for col_2 in gl_final_param:
        if col_1 != col_2:
            lv_correlation = go_data[col_1].corr(go_data[col_2])
            if lv_correlation >= 0.70:
                print(f' Параметр {col_1} коррелирует с парамтером {col_2} : {lv_correlation}')
                gl_final_param.remove(col_2)
display(gl_final_param)

 Параметр BCUT2D_LOGPLOW коррелирует с парамтером BCUT2D_CHGLO : 0.8673416403304163
 Параметр MaxEStateIndex коррелирует с парамтером MaxAbsEStateIndex : 1.0
 Параметр MaxEStateIndex коррелирует с парамтером MinAbsPartialCharge : 0.7537102233342267
 Параметр MaxEStateIndex коррелирует с парамтером MaxPartialCharge : 0.7372088681878384
 Параметр HeavyAtomMolWt коррелирует с парамтером Kappa3 : 0.7408343070027841
 Параметр HeavyAtomMolWt коррелирует с парамтером ExactMolWt : 0.9968205134360127
 Параметр HeavyAtomMolWt коррелирует с парамтером Chi1 : 0.9821608593097768
 Параметр HeavyAtomMolWt коррелирует с парамтером Kappa2 : 0.8930224053498795
 Параметр HeavyAtomMolWt коррелирует с парамтером NumValenceElectrons : 0.96620144667007
 Параметр HeavyAtomMolWt коррелирует с парамтером LabuteASA : 0.9766215252487562
 Параметр HeavyAtomMolWt коррелирует с парамтером Chi0v : 0.9278801908569162
 Параметр HeavyAtomMolWt коррелирует с парамтером MolMR : 0.936038347427541
 Параметр HeavyAtomMolWt к

['BCUT2D_LOGPLOW',
 'MaxEStateIndex',
 'EState_VSA4',
 'RingCount',
 'VSA_EState8',
 'NumRotatableBonds',
 'FpDensityMorgan3',
 'BCUT2D_MWLOW',
 'MolWt',
 'EState_VSA8',
 'VSA_EState7',
 'PEOE_VSA6',
 'MinEStateIndex',
 'SMR_VSA5',
 'BalabanJ',
 'BCUT2D_LOGPHI',
 'MolLogP',
 'Chi4n',
 'VSA_EState4',
 'PEOE_VSA7']

In [171]:
go_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1001 entries, 0 to 1000
Data columns (total 77 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   IC50, mM             1001 non-null   float64
 1   MaxAbsEStateIndex    1001 non-null   float64
 2   MaxEStateIndex       1001 non-null   float64
 3   MinAbsEStateIndex    1001 non-null   float64
 4   MinEStateIndex       1001 non-null   float64
 5   qed                  1001 non-null   float64
 6   SPS                  1001 non-null   float64
 7   MolWt                1001 non-null   float64
 8   HeavyAtomMolWt       1001 non-null   float64
 9   ExactMolWt           1001 non-null   float64
 10  NumValenceElectrons  1001 non-null   float64
 11  MaxPartialCharge     1001 non-null   float64
 12  MinPartialCharge     1001 non-null   float64
 13  MaxAbsPartialCharge  1001 non-null   float64
 14  MinAbsPartialCharge  1001 non-null   float64
 15  FpDensityMorgan1     1001 non-null   float6

In [172]:
X = go_data[gl_final_param]
#X = go_data.drop(columns='IC50, mM')
y = go_data['IC50, mM']

In [173]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

print(f'Train dataset size: {X_train.shape}, {y_train.shape}')
print(f'Test dataset size: {X_test.shape}, {y_test.shape}')

Train dataset size: (700, 20), (700,)
Test dataset size: (301, 20), (301,)


In [174]:
import warnings
warnings.filterwarnings('ignore')

In [175]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor

models = {
    "LinearRegression": (LinearRegression(), 
        {
            'fit_intercept': [True, False]
        }),

    "RidgeRegression": (Ridge(), 
        {
            'alpha': [0.1, 1.0, 10.0, 100.0],  
            'solver': ['auto', 'cholesky', 'sag', 'lsqr']
        }),

    "RandomForestRegressor": (RandomForestRegressor(), 
        {
            'n_estimators': (10, 200, 350),
            'max_depth': (3, 10),
            'min_samples_split': (2, 10, 15, 50)

        })
    
}



In [ ]:
from skopt import BayesSearchCV
from sklearn.metrics import silhouette_score
from sklearn.model_selection import PredefinedSplit

# Перебор моделей
best_global_score = -10
best_model = None
results_report = []


for name, (model, params) in models.items():
    print(f"Обучаем {name}...")

    # Байесовская оптимизация гиперпараметров
    bayes_search = BayesSearchCV(
        estimator=model,
        search_spaces=params,
        n_iter=35,
        cv=10,
        scoring='neg_mean_squared_error',  
        n_jobs=-1,
        random_state=42
    )

    # Обучение модели
    bayes_search.fit(X_train, y_train)

    # 
    score = bayes_search.best_score_  # type: ignore
    results_report.append({"Model": name, "Score": score, "Params": bayes_search.best_params_}) # type: ignore
    
    # Сохраняем абсолютного победителя
    if score > best_global_score:
        best_global_score = score
        best_model = bayes_search.best_estimator_ # type: ignore

# --- АНАЛИЗ ---
print("\n--- Report  ---")
print(pd.DataFrame(results_report))
print(f"\n Лучшая модель: {best_model}")



Обучаем LinearRegression...
Обучаем RidgeRegression...
Обучаем RandomForestRegressor...

--- Report  ---
                   Model     Score  \
0       LinearRegression -3.179982   
1        RidgeRegression -3.161193   
2  RandomForestRegressor -2.367531   

                                              Params  
0                            {'fit_intercept': True}  
1                  {'alpha': 10.0, 'solver': 'lsqr'}  
2  {'max_depth': 9, 'min_samples_split': 10, 'n_e...  

 Лучшая модель: RandomForestRegressor(max_depth=9, min_samples_split=10, n_estimators=200)


In [177]:
# на тестовой выборке 
from sklearn import metrics

y_pred = best_model.predict(X_test)  # type: ignore

print("MAE", metrics.mean_absolute_error(y_test, y_pred))
print("MSE", metrics.mean_squared_error(y_test, y_pred))
print("R2 Score:", best_model.score(X_test, y_test)) # type: ignore

MAE 0.9959177644753252
MSE 1.9723171188589108
R2 Score: 0.4028490963798933
